# Verify `alakazam_top` in the Kaggle grader environment

Runs the agent against itself on Kaggle's native engine (the closest match to the competition
grader) to confirm it **loads and plays legally** before spending a submission. The agent file
already scored live 6× (687–1006), so this is a final load-safety confirmation.

**Setup:** attach the dataset `cha7ura/ptcg-alakazam-top-verify` (Add Input → Datasets).


In [ ]:
import os, sys, importlib.util
AGENT_DIR = '/kaggle/input/ptcg-alakazam-top-verify/agent'
assert os.path.isdir(AGENT_DIR), 'attach the dataset first (Add Input)'
print('agent files:', sorted(os.listdir(AGENT_DIR)))
print('deck.csv lines:', len(open(AGENT_DIR+'/deck.csv').read().split()))


In [ ]:
# confirm native engine loads (x86_64 linux, same as grader)
import platform; print(platform.platform(), platform.machine())
sys.path.insert(0, AGENT_DIR)
from cg.game import battle_start, battle_select, battle_finish
print('engine loaded OK')


In [ ]:
# load the agent's main.py as a module and grab its `agent` entrypoint
spec = importlib.util.spec_from_file_location('agent_main', AGENT_DIR+'/main.py')
m = importlib.util.module_from_spec(spec)
os.chdir(AGENT_DIR)   # main.py loads deck.csv from cwd
spec.loader.exec_module(m)
assert hasattr(m, 'agent'), 'main.py must define agent(obs)'
print('agent entrypoint OK:', m.agent)


In [ ]:
# smoke test: agent vs itself, N games, confirm it completes with no crash/illegal move
import random
DECK = [int(x) for x in open(AGENT_DIR+'/deck.csv').read().split()][:60]
N = 20; wins=[0,0]; errs=0; steps_tot=0
def play(seed):
    obs, sd = battle_start(DECK, DECK)
    if obs is None: return None
    s=0
    try:
        while True:
            cur=obs['current']
            if cur is not None and cur.get('result',-1)!=-1: return cur['result']
            if obs['select'] is None: return None
            a=m.agent(obs); obs=battle_select(a); s+=1
            if s>20000: return None
    finally:
        battle_finish()
random.seed(0)
for g in range(N):
    try:
        r=play(g)
        if r in (0,1): wins[r]+=1
    except Exception as e:
        errs+=1; print('ERR game',g,type(e).__name__,str(e)[:120])
print(f'completed {N-errs}/{N} games, {errs} errors | winner split {wins}')
print('VERDICT:', 'PASS - agent loads & plays legally, safe to submit' if errs==0 else 'FAIL - do not submit')
